In [ ]:
from pathlib import Path
import sys,os,json,subprocess,hashlib,zipfile
from google.colab import files
base=Path('/content/ns64_clean')
blocked=['/content/drive/MyDrive','/content/ns64','/content/imc_ns64_colab_migration.zip','/content/imc_ets_images.zip','/content/private.json']
assert not any(Path(p).exists() for p in blocked),'Use a freshly deleted and reconnected runtime without Drive'
subprocess.run(['nvidia-smi','--query-gpu=name,driver_version','--format=csv,noheader'],check=True)
bundle=Path('/content/imc_ns64_clean_runtime.zip')
if not bundle.exists(): files.upload()
assert hashlib.sha256(bundle.read_bytes()).hexdigest()=='39e9bb97ede83c777a06f385fbe4acb27fc4665ba1556b504dc64000dcafb925','Wrong runtime ZIP'
base.mkdir(exist_ok=True)
with zipfile.ZipFile(bundle) as z:z.extractall(base)
bootstrap=base/'bootstrap'
subprocess.run([sys.executable,'-m','pip','install','--target',str(bootstrap),'uv==0.13.8'],check=True)
setup_env=os.environ.copy()
setup_env.update(PYTHONPATH=str(bootstrap),UV_PYTHON_INSTALL_DIR=str(base/'python'),UV_CACHE_DIR=str(base/'uv_cache'))
uv=[sys.executable,'-m','uv']
subprocess.run(uv+['python','install','3.11.11'],env=setup_env,check=True)
python=base/'venv/bin/python'
if not python.exists():subprocess.run(uv+['venv','--python','3.11.11','--seed',str(base/'venv')],env=setup_env,check=True)
subprocess.run([str(python),'-m','pip','install','setuptools==75.8.0','wheel==0.45.1'],check=True)

subprocess.run([str(python),'-m','pip','install','--no-deps','--extra-index-url','https://download.pytorch.org/whl/cu121','-r',str(base/'requirements.txt')],check=True)


In [ ]:
import urllib.request,shutil
manifest={'assets':json.loads((base/'assets.json').read_text())}
asset_dir=base/'weights'
asset_dir.mkdir(exist_ok=True)
def download(url,path,expected):
    path.parent.mkdir(parents=True,exist_ok=True)
    if path.exists():
        with path.open('rb') as handle: actual=hashlib.file_digest(handle,'sha256').hexdigest()
        assert actual==expected,path
        return
    partial=path.with_suffix(path.suffix+'.partial')
    print('Downloading',path.name,flush=True)
    digest=hashlib.sha256()
    with urllib.request.urlopen(url,timeout=120) as response,partial.open('wb') as output:
        while chunk:=response.read(4*1024**2):
            digest.update(chunk);output.write(chunk)
    assert digest.hexdigest()==expected,path
    partial.replace(path)
for key,a in manifest['assets'].items():
    if key=='dinov2':
        path=asset_dir/'dinov2'/a['filename']
        url='https://huggingface.co/facebook/dinov2-base/resolve/main/pytorch_model.bin'
        for name,h in a['sidecars'].items():
            download('https://huggingface.co/facebook/dinov2-base/resolve/main/'+name,path.parent/name,h)
    else:path=asset_dir/a['filename'];url=a['url']
    download(url,path,a['sha256'])
print('All seven asset hashes verified.')


In [ ]:
result=subprocess.run([str(python),'-I','-c',"import json, sys, os\nfrom pathlib import Path\nimport torch, pycolmap, asmk.hamming, curope, kornia, mast3r.model, dust3r.inference, yaml\nbase=Path('/content/ns64_clean')\nassert torch.__version__=='2.5.1+cu121' and torch.cuda.is_available() and pycolmap.__version__=='3.11.1'\nassets=json.loads((base/'assets.json').read_text())\nsource=base/'source'\naliases=yaml.safe_load((source/'conf/models.yaml').read_text())\nfor key,asset in assets.items():\n    path=base/'weights'/('dinov2' if key=='dinov2' else asset['filename'])\n    assert path.exists()\n    aliases[asset['alias']]={'kernel':str(path),'local':str(path)}\nalias_file=base/'aliases.json'\nalias_file.write_text(json.dumps(aliases))\nos.environ.update(DEFAULT_MODEL_LIST_PATH=str(alias_file),DEFAULT_EXTRA_DIR=str(base/'extra'),MPLBACKEND='Agg',HF_HUB_OFFLINE='1')\nsys.path.insert(0,str(source))\nfrom pipeline import create_pipeline\nfrom config import PipelineConfig\nconfig=PipelineConfig.load_config(source/'conf/pipeline/imc2025/mast3rhybrid/mast3rhybrid-022-c.yaml')\nprint(json.dumps(dict(status='DEPENDENCIES_PASS',python=sys.version,torch=torch.__version__,device=torch.cuda.get_device_name(0),config=config.type)))\n"],cwd=base,capture_output=True,text=True)
print(result.stdout)
print(result.stderr[-4000:])
result.check_returncode()


In [ ]:
from pathlib import Path
from google.colab import files
import subprocess
probe=Path('/content/ns64_clean_vm_precheck.py')
if not probe.exists(): files.upload()
assert probe.exists(),'Upload ns64_clean_vm_precheck.py'
subprocess.run(['/content/ns64_clean/venv/bin/python','-I',str(probe)],check=True)
files.download('/content/imc_ns64_clean_vm_precheck.json')
